In [1]:
import kagglehub
import os, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SR = 22050
N_MELS = 128
N_FFT = 2048
HOP = 512

SEG_SEC = 3.0
SEG_SAMPLES = int(SR * SEG_SEC)

ROOT_OF_GENRES = "/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original"

In [13]:
N_FRAMES = 130  # Фиксированное число кадров по времени

In [4]:
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.
Path to dataset files: /kaggle/input/gtzan-dataset-music-genre-classification


In [15]:
# Индексация файлов с аудио
def list_gztan_files(root_dir):
    items = []
    bad = []
    genres = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    g2i = {g: i for i, g in enumerate(genres)}

    for g in genres:
        folder = os.path.join(root_dir, g)
        for fn in sorted(os.listdir(folder)):
            if not fn.lower().endswith(".wav"):
                continue
            path = os.path.join(folder, fn)
            try:
                sf.info(path)
                items.append((path, g2i[g], g))
            except Exception:
                bad.append(path)
    if bad:
        print(f"[WARN] skipped bad audio files: {len(bad)}")
        print("\n".join(bad[:10]))
    return items, g2i

In [6]:
# Разделение на train/val
def split_by_track(items, test_size=0.2):
    paths = [p for p, y, g in items]
    labels = [y for p, y, g in items]
    train_idx, val_idx = train_test_split(
        np.arange(len(items)),
        test_size=test_size,
        random_state=SEED,
        stratify=labels
    )
    train_items = [items[i] for i in train_idx]
    val_items = [items[i] for i in val_idx]
    return train_items, val_items

In [14]:
class GTZANDataset(Dataset):
    def __init__(self, items, augment=False):
        self.items = items
        self.augment = augment
        self.segments = []
        for path, y, g in items:
            num_seg = 10  # 10 сегментов по 3 сек из 30-секундного трека
            for k in range(num_seg):
                offset = k * SEG_SAMPLES
                self.segments.append((path, y, offset))

    def __len__(self):
        return len(self.segments)

    def _load_segment(self, path, offset):
        wav, _ = librosa.load(path, sr=SR, mono=True)
        if len(wav) < SEG_SAMPLES:
            wav = np.pad(wav, (0, SEG_SAMPLES - len(wav)))
        wav = wav[offset:offset + SEG_SAMPLES]
        return wav

    def _wav_to_mel(self, wav):
      mel = librosa.feature.melspectrogram(
          y=wav, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, power=2.0)
      mel_db = librosa.power_to_db(mel, ref=np.max)

      # Привести к фиксированной длине по времени
      if mel_db.shape[1] < N_FRAMES:
        # Паддинг справа
        pad_width = N_FRAMES - mel_db.shape[1]
        mel_db = np.pad(mel_db, ((0, 0), (0, pad_width)), mode='constant')
      elif mel_db.shape[1] > N_FRAMES:
        # Обрезка
        mel_db = mel_db[:, :N_FRAMES]

      mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
      return mel_db.astype(np.float32)


    def __getitem__(self, idx):
        path, y, offset = self.segments[idx]
        wav = self._load_segment(path, offset)

        # Аугментация (как в оригинале)
        if self.augment:
            if random.random() < 0.3:
                wav = wav + 0.005 * np.random.randn(len(wav))
            if random.random() < 0.3:
                rate = random.uniform(0.9, 1.1)
                wav = librosa.effects.time_stretch(wav, rate=rate)
                wav = librosa.util.fix_length(wav, size=SEG_SAMPLES)

        mel = self._wav_to_mel(wav)
        x = torch.from_numpy(mel).unsqueeze(0)  # [1, N_MELS, T]
        y = torch.tensor(y, dtype=torch.long)
        return x, y


In [8]:
# Чистая CNN
class CNNCleanClassifier(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, 3), padding=(1, 1))
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2))

        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), padding=(1, 1))
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2))

        self.conv3 = nn.Conv2d(64, 128, kernel_size=(3, 3), padding=(1, 1))
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=(2, 2))


        self.conv4 = nn.Conv2d(128, 256, kernel_size=(3, 3), padding=(1, 1))
        self.bn4 = nn.BatchNorm2d(256)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(kernel_size=(2, 2))

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256, n_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.pool4(self.relu4(self.bn4(self.conv4(x))))
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [9]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == y).sum().item()
        total += x.size(0)

    return total_loss / total, total_correct / total

In [10]:
@torch.no_grad()
def eval_model(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for x, y in tqdm(loader, leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == y).sum().item()
        total += x.size(0)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    return total_loss / total, total_correct / total, all_preds, all_labels

In [11]:
def run_experiment(train_items, val_items, epochs=20, batch_size=32, lr=1e-3):
    train_ds = GTZANDataset(train_items, augment=True)
    val_ds = GTZANDataset(val_items, augment=False)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = CNNCleanClassifier(n_classes=10).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_state = None

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_dl, optimizer, criterion)
        va_loss, va_acc, _, _ = eval_model(model, val_dl, criterion)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"epoch {ep:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}")

    # Загрузка лучшей модели
    model.load_state_dict(best_state)
    va_loss, va_acc, preds, labels = eval_model(model, val_dl, criterion)

    print("\nVAL ACC:", va_acc)
    print(classification_report(labels, preds, digits=4))

    return model, va_acc

In [16]:
items, g2i = list_gztan_files(ROOT_OF_GENRES)
train_items, val_items = split_by_track(items, test_size=0.2)

print("tracks:", len(items), "train:", len(train_items), "val:", len(val_items))
print("genres:", list(g2i.keys()))

model, acc = run_experiment(train_items, val_items, epochs=20)
print("\nFinal accuracy:", acc)

[WARN] skipped bad audio files: 1
/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original/jazz/jazz.00054.wav
tracks: 999 train: 799 val: 200
genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


epoch 01 | train loss 1.4637 acc 0.4717 | val loss 1.4276 acc 0.4785


epoch 02 | train loss 1.0960 acc 0.6233 | val loss 1.0712 acc 0.6570


epoch 03 | train loss 0.9439 acc 0.6746 | val loss 1.1254 acc 0.6220


epoch 04 | train loss 0.8466 acc 0.7130 | val loss 1.1090 acc 0.6180


...

epoch 17 | train loss 0.3417 acc 0.8864 | val loss 1.4658 acc 0.6645


epoch 18 | train loss 0.3172 acc 0.8945 | val loss 1.3472 acc 0.6600


epoch 19 | train loss 0.3026 acc 0.8945 | val loss 0.9420 acc 0.7265


epoch 20 | train loss 0.2936 acc 0.9016 | val loss 1.4712 acc 0.6590



VAL ACC: 0.777
              precision    recall  f1-score   support

           0     0.6934    0.9500    0.8017       200
           1     0.7416    0.9900    0.8480       200
           2     0.6255    0.8600    0.7242       200
           3     0.8784    0.6500    0.7471       200
           4     0.8851    0.7700    0.8235       200
           5     0.8190    0.9050    0.8599       200
           6     0.9702    0.8150    0.8859       200
           7     0.7990    0.7950    0.7970       200
           8     0.8400    0.6300    0.7200       200
           9     0.6532    0.4050    0.5000       200

    accuracy                         0.7770      2000
   macro avg     0.7905    0.7770    0.7707      2000
weighted avg     0.7905    0.7770    0.7707      2000


Final accuracy: 0.777


Тренировочная точность стабильно растёт, а тестовая нет, у неё вообще скачки и разрыв между train и val увеличивается. -> Переобучились((
Нужно попробовать:
1) упростить модель (возможна избыточная cnn по сложности, 4 свёрточных блока для такого датасета)
2) добавить ещё регуляризацию + нет аугентации на уровне спектрограмм (только на уровне аудио)
3) проверить разбиение на выборки
4) поднастроить оптимизатор

| Архитектура | Лучшие задачи | Длина аудио | Плюсы | Минусы |
|-----------|-------------|-----------|------|-------|
| **CNN** | - Классификация жанров<br>- Детектирование звуковых событий (сирена, лай)<br>- Распознавание фонем | 1–5 сек | - Быстро обучается<br>- Мало параметров<br>- Простота реализации<br>- Интерпретируемость (визуализация фильтров) | - Не учитывает долгосрочные зависимости<br>- Ограниченное рецептивное поле |
| **CNN + RNN** | - Распознавание речи<br>- Анализ эмоциональной окраски диалога<br>- Классификация музыкальных форм (соната, блюз)<br>- Распознавание команд в речи | 5–15 сек | - Учитывает временную динамику<br>- Моделирует зависимости до десятков секунд<br>- Гибкость для последовательных задач | - Медленнее обучается<br>- Требует больше памяти<br>- Склонен к забыванию (если последовательность очень длинная) |
| **CNN + RNN + Attention** | - Анализ длинных музыкальных треков (с выделением кульминаций)<br>- Распознавание намерений в длинных диалогах<br>- Классификация звуковых сцен с редкими событиями (взрыв, затем тишина) | 10+ сек | - Фокусируется на ключевых моментах<br>- Высокая точность на сложных задачах<br>- Интерпретируемость (визуализация весов attention) | - Самая сложная и медленная<br>- Много параметров<br>- Риск переобучения<br>- Требует качественных данных |
